<a href="https://colab.research.google.com/github/AliZarKazmi/dermassist-llm-MSc_Project-/blob/main/Pi_Mini_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# File Importing and LLM Fine **Tuning**

In [ ]:
import json

file = []
with open("/content/fine_tuning_dataset_skin_disease_3k.json", "r") as f:
    for line in f:
        file.append(json.loads(line))

print(file[1])

{'instruction': 'Give medical details for skin disease labeled MEL', 'input': '', 'output': "1. Description of disease\nMelanoma is primarily caused by DNA damage in skin cells, often due to excessive exposure to ultraviolet (UV) radiation from the sun or tanning beds. Risk factors include fair skin, a history of sunburns, a high number of moles, a family history of melanoma, and weakened immune function.\n\n2. Recommendations\nMelanoma is a serious and potentially life-threatening form of skin cancer. Early detection and treatment are critical to prevent it from spreading to other parts of the body (metastasis). Delaying treatment can significantly reduce survival chances.\n\n3. Diagnosis\nMelanoma is typically diagnosed through a thorough examination of the skin by a healthcare professional, often focusing on unusual moles or skin lesions. If a suspicious area is found, a skin biopsy is performed to remove a small sample of tissue for analysis under a microscope. Additional diagnosti

In [ ]:
!pip install unsloth trl peft accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.9/312.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 544.8/544.8 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.6/213.6 kB 20.8 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
# For GPU check
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

CUDA available: True
GPU: Tesla T4


In [ ]:
pip install unsloth trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch

model_name = "unsloth/Phi-3-mini-4k-instruct-bnb-4bit"

max_seq_length = 2048  # Choose sequence length
dtype = None  # Auto detectiond

# Load model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.8.10: Fast Mistral patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
from datasets import Dataset

def format_prompt(example):
    return f"### Input: {example['input']}\n### Output: {json.dumps(example['output'])}<|endoftext|>"

formatted_data = [format_prompt(item) for item in file]
dataset = Dataset.from_dict({"text": formatted_data})

In [ ]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=64,  # LoRA rank - higher = more capacity, more memory
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=128,  # LoRA scaling factor (usually 2x rank)
    lora_dropout=0,  # Supports any, but = 0 is optimized
    bias="none",     # Supports any, but = "none" is optimized
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized version
    random_state=3407,
    use_rslora=False,  # Rank stabilized LoRA
    loftq_config=None, # LoftQ
)

Unsloth 2025.8.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Training arguments optimized for Unsloth
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,  # Effective batch size = 8
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        save_strategy="epoch",
        save_total_limit=2,
        dataloader_pin_memory=False,
        report_to="none", # Disable Weights & Biases logging
    ),
)

Unsloth: Tokenizing ["text"]:   0%|          | 0/2164 [00:00<?, ? examples/s]

In [ ]:
# Train the model
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,164 | Num Epochs = 3 | Total steps = 813
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 119,537,664 of 3,940,617,216 (3.03% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
25,0.374300
50,0.006000
75,0.005300
100,0.004900
125,0.005200
150,0.005100
175,0.004900
200,0.004700
225,0.004900
250,0.004900


In [ ]:
# Test the fine-tuned model
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# Test prompt
messages =  """### Instruction:
Give medical details for skin disease labeled MEL

### Response:"""

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

# Generate response
outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    use_cache=True,
    temperature=0.7,
    do_sample=True,
    top_p=0.9,
)

# Decode and print
response = tokenizer.batch_decode(outputs)[0]
print(response)

In [ ]:
from peft import PeftModel
from transformers import AutoTokenizer

# Merge LoRA into base model
merged_model = model.merge_and_unload()

# Save merged model
merged_model.save_pretrained("phi3-skin-disease-merged")
tokenizer.save_pretrained("phi3-skin-disease-merged")

/usr/local/lib/python3.11/dist-packages/peft/tuners/lora/bnb.py:348: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


('phi3-skin-disease-merged/tokenizer_config.json',
 'phi3-skin-disease-merged/special_tokens_map.json',
 'phi3-skin-disease-merged/chat_template.jinja',
 'phi3-skin-disease-merged/tokenizer.model',
 'phi3-skin-disease-merged/added_tokens.json',
 'phi3-skin-disease-merged/tokenizer.json')

In [ ]:
model.save_pretrained_gguf("gguf_model", tokenizer, quantization_method="q4_k_m")

Unsloth: You have 1 CPUs. Using `safe_serialization` is 10x slower.
We shall switch to Pytorch saving, which might take 3 minutes and not 30 minutes.
To force `safe_serialization`, set it to `None` instead.
Unsloth: Kaggle/Colab has limited disk space. We need to delete the downloaded
model which will save 4-16GB of disk space, allowing you to save on Kaggle/Colab.
Unsloth: Will remove a cached repo with size 2.3G


Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 4.9 out of 12.67 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


 69%|██████▉   | 22/32 [00:00<00:00, 101.98it/s]
We will save to Disk and not RAM now.
100%|██████████| 32/32 [02:17<00:00,  4.31s/it]


Unsloth: Saving tokenizer... Done.
Unsloth: Saving gguf_model/pytorch_model-00001-of-00003.bin...
Unsloth: Saving gguf_model/pytorch_model-00002-of-00003.bin...
Unsloth: Saving gguf_model/pytorch_model-00003-of-00003.bin...
Done.


Unsloth: Converting mistral model. Can use fast conversion = True.


==((====))==  Unsloth: Conversion from QLoRA to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF 16bits might take 3 minutes.
\        /    [2] Converting GGUF 16bits to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: CMAKE detected. Finalizing some steps for installation.
Unsloth: [1] Converting model at gguf_model into f16 GGUF format.
The output location will be /content/gguf_model/unsloth.F16.gguf
This might take 3 minutes...


Unsloth: Extending gguf_model/tokenizer.model with added_tokens.json.
Originally tokenizer.model is of size (32000).
But we need to extend to sentencepiece vocab size (32011).


INFO:hf-to-gguf:Loading model: gguf_model
INFO:hf-to-gguf:Model architecture: MistralForCausalLM
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:gguf: loading model weight map from 'pytorch_model.bin.index.json'
INFO:hf-to-gguf:gguf: loading model part 'pytorch_model-00001-of-00003.bin'
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {3072, 32064}
INFO:hf-to-gguf:blk.0.attn_q.weight,         torch.float32 --> F16, shape = {3072, 3072}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float32 --> F16, shape = {3072, 3072}
INFO:hf-to-gguf:blk.0.attn_v.weight,         torch.float32 --> F16, shape = {3072, 3072}
INFO:hf-to-gguf:blk.0.attn_output.weight,    torch.float32 --> F16, shape = {3072, 3072}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float32 --> F16, shape = {3072, 8192}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float32 --> F16, shape = {3072, 8192}
INFO:hf-to-gguf:

In [ ]:
# Zip the GGUF model
!zip -r gguf_model.zip gguf_model

# Zip the Hugging Face Transformers model
!zip -r phi3-skin-disease-merged.zip phi3-skin-disease-merged

  adding: gguf_model/ (stored 0%)
  adding: gguf_model/pytorch_model-00001-of-00003.bin (deflated 65%)
  adding: gguf_model/generation_config.json (deflated 32%)
  adding: gguf_model/special_tokens_map.json (deflated 77%)
  adding: gguf_model/pytorch_model.bin.index.json (deflated 95%)
  adding: gguf_model/unsloth.F16.gguf (deflated 59%)
  adding: gguf_model/config.json (deflated 47%)
  adding: gguf_model/tokenizer_config.json (deflated 86%)
  adding: gguf_model/tokenizer.json (deflated 85%)
  adding: gguf_model/tokenizer.model (deflated 55%)
  adding: gguf_model/added_tokens.json (deflated 62%)
  adding: gguf_model/chat_template.jinja (deflated 60%)
  adding: gguf_model/unsloth.Q4_K_M.gguf (deflated 2%)
  adding: gguf_model/pytorch_model-00002-of-00003.bin (deflated 70%)
  adding: gguf_model/pytorch_model-00003-of-00003.bin (deflated 68%)
  adding: phi3-skin-disease-merged/ (stored 0%)
  adding: phi3-skin-disease-merged/generation_config.json (deflated 32%)
  adding: phi3-skin-disease

# **Test the Model**

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Move zip files to Google Drive

!mv gguf_model.zip /content/drive/MyDrive/
!mv phi3-skin-disease-merged.zip /content/drive/MyDrive/

In [ ]:
# Unzip the file

!unzip -q /content/drive/MyDrive/Models/phi3-skin-disease-merged.zip

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load model and tokenizer
model_path = "/content/drive/MyDrive/phi3-skin-disease-merged/phi3-skin-disease-merged"  # Change if in different location

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto"  # Uses GPU if available
)
tokenizer = AutoTokenizer.from_pretrained(model_path)

In [ ]:
from PIL import Image
import numpy as np
from tensorflow.keras.models import load_model
import joblib

model = load_model("/content/skin_cancer_model.h5")
label_encoder = joblib.load("/content/label_encoder.pkl")

def predict_image(image_path):
    img = Image.open(image_path).convert('RGB').resize((100, 75), 3)
    img_array = np.array(img).astype(np.float32) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array)
    predicted_class = np.argmax(prediction, axis=1)[0]
    predicted_label = label_encoder.inverse_transform([predicted_class])[0]
    return predicted_label

In [ ]:
img = "/content/ISIC_0024327.jpg"
model_img_input = predict_image(img)

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step


In [ ]:
prompt = f"""### Instruction:
Give medical details for skin disease labeled bkl

And Answer only based on these points:
1. Description of disease
2. Recommendations
3. Diagnosis
4. Suggest Test
5. Treatment Options
### Response:"""

print(prompt)

### Instruction:
Give medical details for skin disease labeled bkl

And Answer only based on these points:
1. Description of disease
2. Recommendations
3. Diagnosis
4. Suggest Test
5. Treatment Options
### Response:


In [ ]:
# Prepare input
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate output
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,
        do_sample=True,
        top_p=0.95,
        temperature=0.7,
        eos_token_id=tokenizer.eos_token_id
    )

# Decode
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

AttributeError: 'Sequential' object has no attribute 'device'

Final

In [ ]:
pip install --upgrade transformers accelerate


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

llm_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    trust_remote_code=True,
    device_map="auto"
)


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from PIL import Image
import numpy as np
from tensorflow.keras.models import load_model
import joblib
from unsloth import FastLanguageModel

# Load text-generation model
model_path = "/content/drive/MyDrive/phi3-skin-disease-merged/phi3-skin-disease-merged"

llm_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_path, # Loads a LoRA adapter
    max_seq_length=2048, # Use the same sequence length as training
    dtype=None, # Auto detects
    load_in_4bit=True, # Use 4bit quantization as in training
    # for_inference=True, # Removed this argument
)

# Apply inference mode separately
FastLanguageModel.for_inference(llm_model)


# Load image classification model
img_model = load_model("/content/skin_cancer_model.h5")
label_encoder = joblib.load("/content/label_encoder.pkl")

def predict_image(image_path):
    img = Image.open(image_path).convert('RGB').resize((100, 75), 3)
    img_array = np.array(img).astype(np.float32) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = img_model.predict(img_array)
    predicted_class = np.argmax(prediction, axis=1)[0]
    predicted_label = label_encoder.inverse_transform([predicted_class])[0]
    return predicted_label

# Predict skin disease
img = "/content/ISIC_0029360.jpg"
model_img_input = predict_image(img)

# Prepare prompt
prompt = f"""### Instruction:
Give medical details for skin disease labeled {model_img_input}

And Answer only based on these points:
1. Description of disease
2. Recommendations
3. Diagnosis
4. Suggest Test
5. Treatment Options
### Response:"""



==((====))==  Unsloth 2025.8.1: Fast Mistral patching. Transformers: 4.55.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.31.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 496ms/step


In [ ]:
# Prepare input for text model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inputs = tokenizer(prompt, return_tensors="pt").to(device)

# Generate text
with torch.no_grad():
    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=800,
        do_sample=True,
        top_p=0.95,
        temperature=0.7,
        eos_token_id=tokenizer.eos_token_id
    )

# Decode
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

### Instruction:
Give medical details for skin disease labeled bkl

And Answer only based on these points:
1. Description of disease
2. Recommendations
3. Diagnosis
4. Suggest Test
5. Treatment Options
### Response:
Bladder Cancer (Bladder Cancer):

1. **Description of Disease**: Bladder cancer, also known as urothelial carcinoma, is a common malignancy that affects the urinary bladder. It is primarily associated with the urothelial cells that line the interior of the bladder. The disease can present in various stages, from non-invasive papillary urothelial carcinoma to invasive muscle-invasive cancer.

2. **Recommendations**: Patients diagnosed with bladder cancer should consult with a urologist or oncologist for a comprehensive treatment plan. The management plan may include surgery, chemotherapy, radiation therapy, immunotherapy, or a combination of these treatments.

3. **Diagnosis**: Diagnosis of bladder cancer typically involves a combination of urine tests, imaging studies, and 